# Morada Streaming ETL - Apache Kafka & PySpark com Dashboard em Tempo Real

Este notebook demonstra a implementação completa de processamento contínuo em tempo real (Structured Streaming) para a plataforma **Morada**.
Ele executa de forma autocontida a instalação do Kafka, o boot do Broker, o simulador de vendas, as transformações com PySpark e disponibiliza uma **interface gráfica em tempo real**.

In [ ]:
# 1. Instalar Java (Requisito do Kafka e Spark)
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

# 2. Baixar e extrair o Apache Kafka
!wget -q https://archive.apache.org/dist/kafka/3.5.1/kafka_2.12-3.5.1.tgz
!tar -xzf kafka_2.12-3.5.1.tgz

# 3. Instalar PySpark, Kafka-Python, Matplotlib e Seaborn para o Dashboard
!pip install -q pyspark==3.5.0 kafka-python-ng matplotlib seaborn

In [ ]:
import os
import subprocess
import socket
import time

# Configurar JAVA_HOME para o ambiente Linux/Colab
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"

def is_port_open(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(1)
        return s.connect_ex(('127.0.0.1', port)) == 0

# Iniciar o Zookeeper em segundo plano
if not is_port_open(2181):
    print("Iniciando Zookeeper...")
    subprocess.Popen(
        ["./kafka_2.12-3.5.1/bin/zookeeper-server-start.sh", "./kafka_2.12-3.5.1/config/zookeeper.properties"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    while not is_port_open(2181):
        time.sleep(1)
    print("Zookeeper pronto!")
else:
    print("Zookeeper já em execução na porta 2181.")

# Iniciar o Broker Kafka em segundo plano
if not is_port_open(9092):
    print("Iniciando Broker Kafka...")
    subprocess.Popen(
        ["./kafka_2.12-3.5.1/bin/kafka-server-start.sh", "./kafka_2.12-3.5.1/config/server.properties"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    while not is_port_open(9092):
        time.sleep(1)
    print("Broker Kafka pronto!")
else:
    print("Broker Kafka já em execução na porta 9092.")

# Criar o tópico morada-bookings
subprocess.run([
    "./kafka_2.12-3.5.1/bin/kafka-topics.sh", "--create", 
    "--bootstrap-server", "localhost:9092", 
    "--replication-factor", "1", "--partitions", "1", 
    "--topic", "morada-bookings", "--if-not-exists"
])
print("Tópico 'morada-bookings' verificado/configurado com sucesso!")

In [ ]:
import json
import random
import threading
from datetime import datetime
from kafka import KafkaProducer

def run_producer():
    producer = KafkaProducer(
        bootstrap_servers=['localhost:9092'],
        value_serializer=lambda v: json.dumps(v).encode('utf-8')
    )
    nomes_br = ["Ana Souza", "Bruno Lima", "Carlos Silva", "Daniela Costa", "Eduardo Pereira", "Fernanda Ramos", "Gustavo Santos"]
    pagamentos = ["pix", "credit_card", "split_bill"]
    
    while True:
        booking_event = {
            "id": str(random.randint(100000, 999999)),
            "listing_id": str(random.randint(1, 50)),
            "primary_guest_name": random.choice(nomes_br),
            "primary_guest_cpf": f"{random.randint(100,999)}.{random.randint(100,999)}.{random.randint(100,999)}-{random.randint(10,99)}",
            "primary_guest_phone": f"+55 (11) 9{random.randint(8000,9999)}-{random.randint(1000,9999)}",
            "total_price": float(round(random.uniform(300.0, 2000.0), 2)),
            "payment_type": random.choice(pagamentos),
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }
        producer.send('morada-bookings', booking_event)
        time.sleep(random.uniform(2.0, 4.0))

prod_thread = threading.Thread(target=run_producer, daemon=True)
prod_thread.start()
print("Simulador de reservas (Produtor Kafka) rodando em background enviando eventos a cada 2-4s...")

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, regexp_replace, when, expr
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

# Inicializar a sessão Spark com suporte ao Kafka
spark = SparkSession.builder \
    .appName("MoradaStreamingNotebook") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0") \
    .config("spark.sql.shuffle.partitions", "2") \
    .getOrCreate()
    
booking_schema = StructType([
    StructField("id", StringType(), True),
    StructField("listing_id", StringType(), True),
    StructField("primary_guest_name", StringType(), True),
    StructField("primary_guest_cpf", StringType(), True),
    StructField("primary_guest_phone", StringType(), True),
    StructField("total_price", DoubleType(), True),
    StructField("payment_type", StringType(), True),
    StructField("timestamp", StringType(), True)
])

# Leitura streaming do Kafka
df_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "morada-bookings") \
    .option("startingOffsets", "latest") \
    .load()

# Deserialização
df_parsed = df_stream.selectExpr("CAST(value AS STRING) as json_payload") \
    .select(from_json(col("json_payload"), booking_schema).alias("data")) \
    .select("data.*")
    
# Limpeza, LGPD e Cashback
df_clean_cpf = df_parsed.withColumn("cpf_raw", regexp_replace(col("primary_guest_cpf"), r"[.-]", ""))
df_transformed = df_clean_cpf.withColumn(
    "masked_cpf",
    expr("concat('***.', substring(cpf_raw, 4, 3), '.', substring(cpf_raw, 7, 3), '-**')")
).withColumn(
    "phone_clean",
    regexp_replace(col("primary_guest_phone"), r"[^\d+]", "")
).withColumn(
    "earned_cashback",
    when(col("payment_type") == "pix", col("total_price") * 0.02)
    .otherwise(col("total_price") * 0.01)
).select("timestamp", "id", "primary_guest_name", "masked_cpf", "phone_clean", "payment_type", "total_price", "earned_cashback")

# Gravar fluxo em memória para consultar com Spark SQL
query = df_transformed.writeStream \
    .format("memory") \
    .queryName("morada_stream_table") \
    .outputMode("append") \
    .start()

print("Spark Streaming iniciado! Gravando em tempo real na tabela 'morada_stream_table'.")

In [ ]:
# Execute esta célula para abrir a Interface Gráfica / Dashboard de Monitoramento em Tempo Real!
# Pressione o botão de "Stop" do Colab para parar o loop quando desejar.

from IPython.display import display, clear_output
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

# Configurar tema visual do Seaborn
sns.set_theme(style="darkgrid")

while True:
    try:
        # 1. Obter agregações e histórico da tabela em memória do Spark
        df_metrics = spark.sql(
            "SELECT payment_type, count(1) as total_reservas, sum(total_price) as faturamento_total, sum(earned_cashback) as cashback_acumulado FROM morada_stream_table GROUP BY payment_type"
        ).toPandas()
        
        df_latest = spark.sql(
            "SELECT timestamp, id, primary_guest_name, masked_cpf, phone_clean, payment_type, total_price, earned_cashback FROM morada_stream_table ORDER BY timestamp DESC LIMIT 5"
        ).toPandas()
        
        # 2. Renderizar interface apenas se já houver registros
        if not df_metrics.empty:
            clear_output(wait=True)
            
            # Criar container de figuras
            fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
            
            # Cores customizadas para realçar o PIX
            colors = ["#00b4d8" if pt.lower() == "pix" else "#9b5de5" if pt.lower() == "split_bill" else "#f15bb5" for pt in df_metrics["payment_type"]]
            
            # Gráfico Esquerdo: Receita total acumulada por método de pagamento
            sns.barplot(x="payment_type", y="faturamento_total", data=df_metrics, ax=axes[0], palette=colors, hue="payment_type", legend=False)
            axes[0].set_title("Faturamento Total Acumulado (R$)", fontsize=13, fontweight="bold", pad=12)
            axes[0].set_ylabel("Faturamento Total (R$)")
            axes[0].set_xlabel("Método de Pagamento")
            
            # Gráfico Direito: Reservas por método de pagamento (Donut Chart)
            axes[1].pie(
                df_metrics["total_reservas"], 
                labels=df_metrics["payment_type"].str.upper(), 
                autopct='%1.1f%%', 
                startangle=90, 
                colors=colors, 
                wedgeprops=dict(width=0.4, edgecolor='w')
            )
            axes[1].set_title("Distribuição do Volume de Reservas", fontsize=13, fontweight="bold", pad=12)
            
            # Ajustar layout do plot
            plt.tight_layout()
            display(plt.gcf())
            plt.close() # Evita duplicação do plot no output
            
            # 3. Exibir Painel de Indicadores Gerais (KPI Cards)
            total_reservas = df_metrics["total_reservas"].sum()
            faturamento_total = df_metrics["faturamento_total"].sum()
            cashback_total = df_metrics["cashback_acumulado"].sum()
            
            print("╔═══════════════════════════════════════════════════════════════════════════════╗")
            print(f"║                 PAINEL MORADA MONITORING - STREAMING EM TEMPO REAL            ║")
            print("╠══════════════════════════════╦══════════════════════════════╦═════════════════╣")
            print(f"║  Reservas Processadas: {total_reservas:<5} ║  Faturamento: R$ {faturamento_total:<11,.2f} ║  Cashback: R$ {cashback_total:<5,.2f}║")
            print("╚══════════════════════════════╩══════════════════════════════╩═════════════════╝")
            
            # 4. Mostrar os últimos eventos transformados (LGPD protegida)
            print("\n🔍 Últimas 5 Reservas Sanitizadas (Prontas para Ingestão Silver/Gold):")
            display(df_latest)
            
        else:
            print("Aguardando os primeiros eventos do simulador chegarem no Kafka...")
            
        time.sleep(3) # Taxa de atualização do dashboard
        
    except KeyboardInterrupt:
        print("\n[INFO] Interface gráfica encerrada pelo usuário.")
        break
    except Exception as e:
        print(f"Aguardando ativação do stream... ({e})")
        time.sleep(3)